# Customer Segmentation & Churn Pattern Analytics in European Banking

This notebook performs the full EDA and segmentation-driven churn analysis for the project.

**Dataset:** `European_Bank.csv`

**Important:** The project brief specifies segment labels but does not give exact numeric cutoffs for all segments. The notebook documents reproducible choices for age, credit score, tenure and balance.


In [ ]:
# Install/import libraries
!pip -q install pandas numpy scipy matplotlib seaborn plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from google.colab import files
from IPython.display import display

print("Libraries loaded.")


## 1. Upload and load the CSV

In [ ]:
# Upload the CSV directly into Google Colab
uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# Data quality checks
print("Data information:")
df.info()

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nExited distribution:")
print(df["Exited"].value_counts())
print(df["Exited"].value_counts(normalize=True).mul(100).round(2))


## 2. Create customer segments

Reproducible rules used in this project:

- Age: `<30`, `30–45`, `46–60`, `60+`
- Credit score: `<600` Low, `600–699` Medium, `700+` High
- Tenure: `0–3` New, `4–6` Mid-term, `7–10` Long-term
- Balance: zero-balance, low-balance and high-balance using the median positive balance

The brief did not specify numeric cutoffs for every category, so the rules above are explicitly documented rather than hidden.


In [ ]:
df["Age_Segment"] = pd.cut(
    df["Age"],
    bins=[-np.inf, 29, 45, 60, np.inf],
    labels=["<30", "30–45", "46–60", "60+"],
    include_lowest=True
)

df["Credit_Score_Band"] = pd.cut(
    df["CreditScore"],
    bins=[-np.inf, 599, 699, np.inf],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

df["Tenure_Group"] = pd.cut(
    df["Tenure"],
    bins=[-np.inf, 3, 6, np.inf],
    labels=["New", "Mid-term", "Long-term"],
    include_lowest=True
)

median_positive_balance = df.loc[df["Balance"] > 0, "Balance"].median()

df["Balance_Segment"] = np.select(
    [
        df["Balance"].eq(0),
        (df["Balance"] > 0) & (df["Balance"] < median_positive_balance)
    ],
    ["Zero-balance", "Low-balance"],
    default="High-balance"
)

df["High_Value"] = df["Balance_Segment"].eq("High-balance")

print("Median positive balance used:", round(median_positive_balance, 2))
display(df.head())


## 3. Overall churn

In [ ]:
overall_churn = df["Exited"].mean() * 100
print(f"Overall churn rate: {overall_churn:.2f}%")

status = (
    df["Exited"]
    .map({0: "Retained", 1: "Churned"})
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Customers")
)
display(status)

plt.figure(figsize=(8, 5))
sns.barplot(data=status, x="Status", y="Customers")
plt.title("Retained vs Churned Customers")
plt.xlabel("Customer status")
plt.ylabel("Customers")
plt.show()


## 4. Geography-wise churn

In [ ]:
def segment_summary(data, column):
    out = (
        data.groupby(column, observed=False)
        .agg(
            Customers=("Exited", "size"),
            Churners=("Exited", "sum"),
            Churn_Rate=("Exited", "mean")
        )
        .reset_index()
    )
    out["Churn_Rate_Pct"] = out["Churn_Rate"] * 100
    out["Churn_Contribution_Pct"] = out["Churners"] / data["Exited"].sum() * 100
    return out

geo = segment_summary(df, "Geography")
geo["Geographic_Risk_Index"] = geo["Churn_Rate"] / df["Exited"].mean() * 100

display(geo.round(3))

plt.figure(figsize=(8, 5))
sns.barplot(data=geo, x="Geography", y="Churn_Rate_Pct")
plt.title("Churn Rate by Geography")
plt.xlabel("Geography")
plt.ylabel("Churn rate (%)")
plt.show()


## 5. Age and tenure

In [ ]:
age = segment_summary(df, "Age_Segment")
tenure = segment_summary(df, "Tenure_Group")

print("Age segmentation")
display(age.round(3))

print("Tenure segmentation")
display(tenure.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=age, x="Age_Segment", y="Churn_Rate_Pct", ax=axes[0])
axes[0].set_title("Churn Rate by Age Segment")
axes[0].set_xlabel("Age segment")
axes[0].set_ylabel("Churn rate (%)")

sns.barplot(data=tenure, x="Tenure_Group", y="Churn_Rate_Pct", ax=axes[1])
axes[1].set_title("Churn Rate by Tenure Group")
axes[1].set_xlabel("Tenure group")
axes[1].set_ylabel("Churn rate (%)")

plt.tight_layout()
plt.show()


## 6. Credit score and balance

In [ ]:
credit = segment_summary(df, "Credit_Score_Band")
balance = segment_summary(df, "Balance_Segment")

print("Credit score bands")
display(credit.round(3))

print("Balance segments")
display(balance.round(3))

high_value = df[df["High_Value"]]
print(f"High-value customer count: {len(high_value):,}")
print(f"High-value churn ratio: {high_value['Exited'].mean() * 100:.2f}%")
print(f"High-value churners: {int(high_value['Exited'].sum()):,}")


## 7. Gender, activity and number of products

In [ ]:
gender = segment_summary(df, "Gender")
products = segment_summary(df, "NumOfProducts")

active_rate = df.loc[df["IsActiveMember"] == 1, "Exited"].mean() * 100
inactive_rate = df.loc[df["IsActiveMember"] == 0, "Exited"].mean() * 100
engagement_drop = inactive_rate - active_rate

print("Gender")
display(gender.round(3))

print("Products")
display(products.round(3))

print(f"Active-member churn rate: {active_rate:.2f}%")
print(f"Inactive-member churn rate: {inactive_rate:.2f}%")
print(f"Engagement Drop Indicator: {engagement_drop:.2f} percentage points")


## 8. Statistical association tests

In [ ]:
def association_test(data, column):
    table = pd.crosstab(data[column], data["Exited"])
    chi2, p, dof, expected = stats.chi2_contingency(table)
    n = table.values.sum()
    v = np.sqrt((chi2 / n) / min(table.shape[0] - 1, table.shape[1] - 1))
    return {
        "Variable": column,
        "Chi_Square": chi2,
        "P_Value": p,
        "Degrees_of_Freedom": dof,
        "Cramers_V": v
    }

association_rows = []
for col in [
    "Geography", "Gender", "Age_Segment", "Credit_Score_Band",
    "Tenure_Group", "Balance_Segment", "IsActiveMember", "NumOfProducts"
]:
    association_rows.append(association_test(df, col))

association_results = pd.DataFrame(association_rows)
display(association_results.round(5))


## 9. Numeric correlations with churn

In [ ]:
numeric_cols = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary"
]

numeric_corr = (
    df[numeric_cols + ["Exited"]]
    .corr(numeric_only=True)["Exited"]
    .drop("Exited")
    .sort_values(ascending=False)
)

display(numeric_corr.to_frame("Pearson_Correlation_with_Exited").round(4))


## 10. High-value customer financial exposure

In [ ]:
total_balance = df["Balance"].sum()
churned_balance = df.loc[df["Exited"] == 1, "Balance"].sum()
high_value_churned_balance = df.loc[
    (df["Exited"] == 1) & (df["High_Value"]),
    "Balance"
].sum()

print(f"Total balance: {total_balance:,.2f}")
print(f"Churned-customer balance exposure: {churned_balance:,.2f}")
print(f"Churned balance share: {churned_balance / total_balance * 100:.2f}%")
print(f"High-value churned balance exposure: {high_value_churned_balance:,.2f}")


## 11. Final interpretation checklist

- Report effect sizes together with p-values.
- Large datasets can make tiny differences statistically detectable.
- High churn rates in small segments should be treated cautiously.
- Use the segmentation thresholds documented above for reproducibility.
- Do not interpret these observational results as causal evidence.
